# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata safely via the interface (not via subscripting or as dict)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, their associated fields, and IDs.

We inspect the record sets and their fields using their `@id` values for referencing.


In [ ]:
# List all record sets, fields, and columns by their @id.
print('Record Sets:')
for recset in dataset.record_sets:
    print(f"- RecordSet @id: {recset.id}")
    print(f"  name: {recset.name}")
    print(f"  Description: {recset.description}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      - Column @id: {col.id} (name: {col.name})")
    print()

## 3. Data Extraction
Load data from the available record sets into DataFrames for analysis. We use the record set and field `@id`s identified above.

In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [recset.id for recset in dataset.record_sets]

dataframes = {}
for rec_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data (top 3 rows):\n{df.head(3)}\n")
# Select the first record set for demonstration below
main_record_set = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping.

In the following, we select a numeric field (by its @id), then filter, normalize, and group the data accordingly.


In [ ]:
if main_record_set is not None:
    main_df = dataframes[main_record_set]
    print(f"Available columns for {main_record_set}: {main_df.columns.tolist()}")
    # Heuristically search for a likely numeric field by common mass-spectrometry names -- fallback to the first numeric-like column
    import numpy as np

    # Look for likely float/int columns
    possible_numeric = [c for c in main_df.columns if any(w in c.lower() for w in ['mw', 'molecular', 'weight', 'count', 'normalized', 'intensity', 'abundance', 'value', 'score'])]
    numeric_field = possible_numeric[0] if possible_numeric else None

    found_numeric_field = False
    if numeric_field is not None:
        # Coerce to numeric
        vals = pd.to_numeric(main_df[numeric_field], errors='coerce')
        if vals.notnull().sum() > 0:
            numeric_field_id = numeric_field
            found_numeric_field = True
    if not found_numeric_field:
        # Fallback to the first column with numeric values
        for col in main_df.columns:
            vals = pd.to_numeric(main_df[col], errors='coerce')
            if vals.notnull().sum() > 0:
                numeric_field_id = col
                found_numeric_field = True
                break
    if found_numeric_field:
        print(f"Using numeric field: {numeric_field_id}")
        # Filter records: greater than mean (for demo)
        vals = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
        threshold = vals.mean()
        filtered_df = main_df[vals > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)}):")
        print(filtered_df[[numeric_field_id]].head())
        # Normalization (z-score)
        filtered_df[numeric_field_id + '_normalized'] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - vals.mean()) / vals.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
        # Group by a likely categorical variable if found
        possible_group_fields = [c for c in main_df.columns if c not in [numeric_field_id] and main_df[c].nunique() < 12]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field}:")
            print(grouped_df)
    else:
        print("No numeric field available for demonstration.")
else:
    print("No record sets present in the dataset.")

## 5. Visualization
Visualize a distribution or relationship in the main record set (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set is not None and found_numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("Not enough information to visualize numeric fields.")

## 6. Conclusion
In this notebook, we explored the Mass Spectrometry dataset by loading the Croissant schema and data records with `mlcroissant`. We:
- Inspected available record sets, fields, and columns via their `@id`s
- Loaded the records into pandas DataFrames
- Filtered, normalized, and grouped data based on automatically detected numeric and categorical fields
- Visualized a sample distribution

This structured approach enables efficient exploration and processing of FAIR datasets described with Croissant, where all relevant entities are referenced by their `@id` for reproducible, standards-based analytics.